In [3]:
import pandas as pd

In [1]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [7]:
INPUT_PATH = "../../data/authorization_preprocessed.csv"

df = pd.read_csv(INPUT_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully
Shape: (15180, 22)


,authorization_id,member_id,provider_id,facility_id,service_date,authorization_date,service_type,procedure_code,specialty,place_of_service,...,submission_channel,authorization_status,processing_time_hours,missing_document_count,resubmission_count,appeal_flag,claim_submitted_flag,claim_amount,anomaly_label,anomaly_type
0,AUTH10142,MEM01555,PRV0440,FAC0110,2025-09-23,2025-09-16,Inpatient,PROC004,Neurology,Outpatient,...,Portal,Approved,21.65,0,1,0,1,2235.76,0,Normal
1,AUTH04414,MEM00462,PRV0253,FAC0082,2026-04-08,2026-04-10,CT Scan,PROC001,Oncology,Emergency,...,Portal,Pending,NaN,2,0,0,1,51259.83,1,Extreme_Claim_Amount
2,AUTH06494,MEM01664,PRV0171,FAC0086,2026-03-15,2026-03-13,Specialist Visit,PROC003,Surgery,Home,...,Fax,Approved,22.21,2,0,0,1,3477.73,0,Normal
3,AUTH08050,MEM01242,PRV0493,FAC0030,2026-05-21,2026-05-22,Surgery,PROC004,Neurology,Outpatient,...,EDI,Approved,21.99,1,0,0,1,1985.53,0,Normal
4,AUTH07435,MEM02182,PRV0330,FAC0051,2025-10-22,2025-10-16,DME,PROC001,Surgery,Home,...,EDI,Approved,55.01,1,1,0,1,5648.02,0,Normal


In [8]:
print("COLUMN NAMES:")
print(df.columns.tolist())

print("\nDATA TYPES:")
print(df.dtypes)

COLUMN NAMES:
['authorization_id', 'member_id', 'provider_id', 'facility_id', 'service_date', 'authorization_date', 'service_type', 'procedure_code', 'specialty', 'place_of_service', 'urgency', 'authorization_type', 'submission_channel', 'authorization_status', 'processing_time_hours', 'missing_document_count', 'resubmission_count', 'appeal_flag', 'claim_submitted_flag', 'claim_amount', 'anomaly_label', 'anomaly_type']

DATA TYPES:
authorization_id           object
member_id                  object
provider_id                object
facility_id                object
service_date               object
authorization_date         object
service_type               object
procedure_code             object
specialty                  object
place_of_service           object
urgency                    object
authorization_type         object
submission_channel         object
authorization_status       object
processing_time_hours     float64
missing_document_count      int64
resubmission_count  

In [9]:
print("\nMISSING VALUES:")
print(df.isnull().sum())

print("\nDUPLICATE ROWS:")
print(df.duplicated().sum())

print("\nDATASET INFO:")
df.info()


MISSING VALUES:
authorization_id            0
member_id                 151
provider_id                 0
facility_id               303
service_date               74
authorization_date         46
service_type                0
procedure_code            225
specialty                 305
place_of_service          123
urgency                   182
authorization_type          0
submission_channel        227
authorization_status        0
processing_time_hours     226
missing_document_count      0
resubmission_count          0
appeal_flag                 0
claim_submitted_flag        0
claim_amount              151
anomaly_label               0
anomaly_type                0
dtype: int64

DUPLICATE ROWS:
0

DATASET INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15180 entries, 0 to 15179
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   authorization_id        15180 non-null  object 
 1   mem

In [10]:
required_columns = [
    "authorization_id",
    "member_id",
    "provider_id",
    "authorization_date",
    "service_date",
    "service_type",
    "procedure_code",
    "place_of_service",
    "urgency",
    "authorization_status",
    "submission_channel",
    "authorization_type",
    "processing_time_hours",
    "missing_document_count",
    "resubmission_count"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if len(missing_columns) == 0:
    print("SCHEMA VALIDATION: PASSED")
else:
    print("SCHEMA VALIDATION: FAILED")
    print("Missing Columns:", missing_columns)

unexpected_columns = [
    col for col in df.columns
    if col not in required_columns
    and col not in ["anomaly_label", "anomaly_type"]
]

print("\nUnexpected Columns:")
print(unexpected_columns)

SCHEMA VALIDATION: PASSED

Unexpected Columns:
['facility_id', 'specialty', 'appeal_flag', 'claim_submitted_flag', 'claim_amount']


In [11]:
if "anomaly_label" in df.columns:
    y_true = df["anomaly_label"].copy()

    print("Ground truth found.")
    print(y_true.value_counts())

else:
    y_true = None
    print("No anomaly_label found.")
    print("Running in unsupervised real-world mode.")

Ground truth found.
anomaly_label
0    12901
1     2279
Name: count, dtype: int64


In [12]:
df["authorization_date"] = pd.to_datetime(
    df["authorization_date"],
    errors="coerce"
)

df["service_date"] = pd.to_datetime(
    df["service_date"],
    errors="coerce"
)

print("Date conversion completed.")

print("\nMissing authorization dates:")
print(df["authorization_date"].isnull().sum())

print("\nMissing service dates:")
print(df["service_date"].isnull().sum())

Date conversion completed.

Missing authorization dates:
46

Missing service dates:
74


In [13]:
required_fields = [
    "authorization_id",
    "member_id",
    "provider_id",
    "authorization_date",
    "service_date",
    "service_type",
    "procedure_code",
    "authorization_status"
]

for col in required_fields:
    df[f"rule_missing_{col}"] = (
        df[col].isnull()
    ).astype(int)

print("Missing field rules completed.")

Missing field rules completed.


In [14]:
df["rule_invalid_authorization_id"] = (
    ~df["authorization_id"]
    .fillna("")
    .astype(str)
    .str.match(r"^AUTH\d+$")
).astype(int)

df["rule_invalid_member_id"] = (
    ~df["member_id"]
    .fillna("")
    .astype(str)
    .str.match(r"^MEM\d+$")
).astype(int)

df["rule_invalid_provider_id"] = (
    ~df["provider_id"]
    .fillna("")
    .astype(str)
    .str.match(r"^PRV\d+$")
).astype(int)

print("ID format rules completed.")

ID format rules completed.


In [15]:
today = pd.Timestamp.today().normalize()

df["rule_future_authorization_date"] = (
    df["authorization_date"] > today
).astype(int)

df["rule_future_service_date"] = (
    df["service_date"] > today
).astype(int)

df["rule_invalid_date_sequence"] = (
    (df["service_date"] < df["authorization_date"]) &
    (df["authorization_type"] != "Retrospective")
).astype(int)

print("Date rules completed.")

Date rules completed.


In [16]:
valid_urgency = [
    "Routine",
    "Urgent",
    "Emergency"
]

valid_status = [
    "Approved",
    "Denied",
    "Pending"
]

valid_channels = [
    "Portal",
    "EDI",
    "Fax",
    "Phone"
]

valid_auth_types = [
    "Initial",
    "Extension",
    "Concurrent",
    "Retrospective"
]

df["rule_invalid_urgency"] = (
    ~df["urgency"].isin(valid_urgency)
).astype(int)

df["rule_invalid_status"] = (
    ~df["authorization_status"].isin(valid_status)
).astype(int)

df["rule_invalid_submission_channel"] = (
    ~df["submission_channel"].isin(valid_channels)
).astype(int)

df["rule_invalid_authorization_type"] = (
    ~df["authorization_type"].isin(valid_auth_types)
).astype(int)

print("Category validation rules completed.")

Category validation rules completed.


In [17]:
df["rule_negative_processing_time"] = (
    df["processing_time_hours"] < 0
).astype(int)

df["rule_excessive_processing_time"] = (
    df["processing_time_hours"] > 720
).astype(int)

df["rule_negative_missing_documents"] = (
    df["missing_document_count"] < 0
).astype(int)

df["rule_negative_resubmissions"] = (
    df["resubmission_count"] < 0
).astype(int)

print("Numeric rules completed.")

Numeric rules completed.


In [18]:
df["rule_approved_high_missing_docs"] = (
    (df["authorization_status"] == "Approved") &
    (df["missing_document_count"] >= 5)
).astype(int)

print("Business consistency rule completed.")

Business consistency rule completed.


In [19]:
df["rule_duplicate_record"] = (
    df.duplicated(
        subset=[
            "member_id",
            "provider_id",
            "authorization_date",
            "service_date",
            "procedure_code"
        ],
        keep=False
    )
).astype(int)

print("Duplicate rule completed.")

Duplicate rule completed.


In [20]:
df.drop(
    columns=["rule_service_procedure_mismatch"],
    errors="ignore",
    inplace=True
)

rule_columns = [
    col for col in df.columns
    if col.startswith("rule_")
    and col not in [
        "rule_score",
        "rule_anomaly",
        "rule_risk_score",
        "rule_severity"
    ]
]

df["rule_score"] = df[rule_columns].sum(axis=1)

df["rule_anomaly"] = (
    df["rule_score"] > 0
).astype(int)

print("Total Rule-Based Anomalies:")
print(df["rule_anomaly"].sum())

print("\nRule Score Distribution:")
print(df["rule_score"].value_counts().sort_index())

Total Rule-Based Anomalies:
4251

Rule Score Distribution:
rule_score
0    10929
1     3658
2      539
3       51
4        3
Name: count, dtype: int64


In [21]:
rule_summary = (
    df[rule_columns]
    .sum()
    .sort_values(ascending=False)
)

print("TOP RULE VIOLATIONS:")
print(rule_summary)

TOP RULE VIOLATIONS:
rule_invalid_date_sequence           2431
rule_duplicate_record                 360
rule_invalid_member_id                327
rule_approved_high_missing_docs       288
rule_invalid_provider_id              229
rule_invalid_submission_channel       227
rule_missing_procedure_code           225
rule_invalid_urgency                  182
rule_missing_member_id                151
rule_invalid_authorization_id         113
rule_future_service_date              112
rule_missing_service_date              74
rule_future_authorization_date         61
rule_missing_authorization_date        46
rule_negative_processing_time          45
rule_negative_missing_documents        30
rule_missing_authorization_status       0
rule_missing_service_type               0
rule_invalid_status                     0
rule_invalid_authorization_type         0
rule_excessive_processing_time          0
rule_negative_resubmissions             0
rule_missing_provider_id                0
rule_missing_

In [23]:
import numpy as np

In [24]:
df["rule_severity"] = np.select(
    [
        df["rule_score"] == 0,
        df["rule_score"] == 1,
        df["rule_score"].between(2, 3),
        df["rule_score"] >= 4
    ],
    [
        "Normal",
        "Low",
        "Medium",
        "Critical"
    ],
    default="Normal"
)

print("RULE SEVERITY:")
print(df["rule_severity"].value_counts())

RULE SEVERITY:
rule_severity
Normal      10929
Low          3658
Medium        590
Critical        3
Name: count, dtype: int64


In [25]:
df["authorization_to_service_days"] = (
    df["service_date"] - df["authorization_date"]
).dt.days

In [26]:
df["provider_avg_processing_time"] = (
    df.groupby("provider_id")["processing_time_hours"]
    .transform("mean")
)

df["provider_avg_resubmission"] = (
    df.groupby("provider_id")["resubmission_count"]
    .transform("mean")
)

df["provider_avg_missing_docs"] = (
    df.groupby("provider_id")["missing_document_count"]
    .transform("mean")
)

In [27]:
df["processing_time_provider_deviation"] = (
    df["processing_time_hours"] -
    df["provider_avg_processing_time"]
).abs()

print("Feature engineering completed.")

Feature engineering completed.


In [28]:
ml_features = [
    "processing_time_hours",
    "missing_document_count",
    "resubmission_count",
    "authorization_to_service_days",
    "provider_avg_processing_time",
    "provider_avg_resubmission",
    "provider_avg_missing_docs",
    "processing_time_provider_deviation"
]

X = df[ml_features].copy()

print("ML Feature Shape:", X.shape)

X.head()

ML Feature Shape: (15180, 8)


,processing_time_hours,missing_document_count,resubmission_count,authorization_to_service_days,provider_avg_processing_time,provider_avg_resubmission,provider_avg_missing_docs,processing_time_provider_deviation
0,21.65,0,1,7.0,30.227838,0.263158,0.473684,8.577838
1,NaN,2,0,-2.0,32.761200,0.846154,0.653846,NaN
2,22.21,2,0,2.0,37.141071,0.535714,0.678571,14.931071
3,21.99,1,0,-1.0,26.192800,0.920000,0.560000,4.202800
4,55.01,1,1,6.0,32.276190,0.738095,0.952381,22.733810


In [29]:
for col in ml_features:
    X[f"{col}_missing_flag"] = (
        X[col].isnull()
    ).astype(int)

print("Missing value flags created.")

X.head()

Missing value flags created.


,processing_time_hours,missing_document_count,resubmission_count,authorization_to_service_days,provider_avg_processing_time,provider_avg_resubmission,provider_avg_missing_docs,processing_time_provider_deviation,processing_time_hours_missing_flag,missing_document_count_missing_flag,resubmission_count_missing_flag,authorization_to_service_days_missing_flag,provider_avg_processing_time_missing_flag,provider_avg_resubmission_missing_flag,provider_avg_missing_docs_missing_flag,processing_time_provider_deviation_missing_flag
0,21.65,0,1,7.0,30.227838,0.263158,0.473684,8.577838,0,0,0,0,0,0,0,0
1,NaN,2,0,-2.0,32.761200,0.846154,0.653846,NaN,1,0,0,0,0,0,0,1
2,22.21,2,0,2.0,37.141071,0.535714,0.678571,14.931071,0,0,0,0,0,0,0,0
3,21.99,1,0,-1.0,26.192800,0.920000,0.560000,4.202800,0,0,0,0,0,0,0,0
4,55.01,1,1,6.0,32.276190,0.738095,0.952381,22.733810,0,0,0,0,0,0,0,0


In [30]:
for col in ml_features:
    
    median_value = X[col].median()

    if pd.isna(median_value):
        median_value = 0

    X[col] = X[col].fillna(median_value)

X = X.fillna(0)

print("Remaining missing values:")
print(X.isnull().sum().sum())

Remaining missing values:
0


In [31]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Feature scaling completed.")
print("Scaled Shape:", X_scaled.shape)

Feature scaling completed.
Scaled Shape: (15180, 16)


In [32]:
iso_model = IsolationForest(
    n_estimators=300,
    contamination=0.08,
    random_state=42
)

iso_prediction = iso_model.fit_predict(X_scaled)

df["ml_anomaly"] = (
    iso_prediction == -1
).astype(int)

print("ML ANOMALY DISTRIBUTION:")
print(df["ml_anomaly"].value_counts())

ML ANOMALY DISTRIBUTION:
ml_anomaly
0    13965
1     1215
Name: count, dtype: int64


In [33]:
df["ml_anomaly_score"] = (
    -iso_model.score_samples(X_scaled)
)

score_scaler = MinMaxScaler()

df["ml_risk_score"] = (
    score_scaler.fit_transform(
        df[["ml_anomaly_score"]]
    )
)

df[
    [
        "authorization_id",
        "ml_anomaly",
        "ml_anomaly_score",
        "ml_risk_score"
    ]
].head()

,authorization_id,ml_anomaly,ml_anomaly_score,ml_risk_score
0,AUTH10142,0,0.353775,0.066832
1,AUTH04414,1,0.595898,0.706137
2,AUTH06494,0,0.344843,0.043246
3,AUTH08050,0,0.344035,0.041114
4,AUTH07435,0,0.352384,0.063159


In [34]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df["behavior_cluster"] = (
    kmeans.fit_predict(X_scaled)
)

print("CLUSTER DISTRIBUTION:")
print(df["behavior_cluster"].value_counts())

CLUSTER DISTRIBUTION:
behavior_cluster
1    14253
0      701
2      226
Name: count, dtype: int64


In [35]:
cluster_summary = (
    df.groupby("behavior_cluster")[ml_features]
    .mean()
)

cluster_summary

,processing_time_hours,missing_document_count,resubmission_count,authorization_to_service_days,provider_avg_processing_time,provider_avg_resubmission,provider_avg_missing_docs,processing_time_provider_deviation
behavior_cluster,,,,,,,,
0,200.786648,0.601997,2.007133,1.628407,79.165564,2.100846,0.690265,150.787162
1,26.329813,0.748614,0.702378,2.228674,32.311457,0.695472,0.741990,14.690807
2,NaN,0.584071,0.623894,2.657778,34.424482,0.768785,0.728060,NaN


In [36]:
distances = kmeans.transform(X_scaled)

assigned_clusters = df["behavior_cluster"].values

df["cluster_distance"] = distances[
    np.arange(len(df)),
    assigned_clusters
]

cluster_distance_scaler = MinMaxScaler()

df["cluster_risk_score"] = (
    cluster_distance_scaler.fit_transform(
        df[["cluster_distance"]]
    )
)

print("Cluster risk score created.")

Cluster risk score created.


In [37]:
max_rule_score = df["rule_score"].max()

if max_rule_score > 0:
    df["rule_risk_score"] = (
        df["rule_score"] / max_rule_score
    )
else:
    df["rule_risk_score"] = 0

In [38]:
df["final_risk_score"] = (
    0.35 * df["rule_risk_score"] +
    0.45 * df["ml_risk_score"] +
    0.20 * df["cluster_risk_score"]
)

print(df["final_risk_score"].describe())

count    15180.000000
mean         0.093916
std          0.102580
min          0.001203
25%          0.022200
50%          0.047833
75%          0.125130
max          0.796021
Name: final_risk_score, dtype: float64


In [39]:
df["final_severity"] = np.select(
    [
        df["final_risk_score"] < 0.30,
        df["final_risk_score"] < 0.55,
        df["final_risk_score"] < 0.75
    ],
    [
        "Normal",
        "Warning",
        "High"
    ],
    default="Critical"
)

print("FINAL SEVERITY:")
print(df["final_severity"].value_counts())

FINAL SEVERITY:
final_severity
Normal      14040
Warning      1119
High           20
Critical        1
Name: count, dtype: int64


In [40]:
sla_map = {
    "Normal": "No Action",
    "Warning": "Review within 48 Hours",
    "High": "Review within 24 Hours",
    "Critical": "Immediate Review within 4 Hours"
}

df["sla"] = df["final_severity"].map(sla_map)

print("SLA DISTRIBUTION:")
print(df["sla"].value_counts())

SLA DISTRIBUTION:
sla
No Action                          14040
Review within 48 Hours              1119
Review within 24 Hours                20
Immediate Review within 4 Hours        1
Name: count, dtype: int64


In [41]:
if y_true is not None:
    
    print("ISOLATION FOREST EVALUATION\n")

    print(
        classification_report(
            y_true,
            df["ml_anomaly"],
            digits=3
        )
    )

    print("CONFUSION MATRIX:")

    print(
        confusion_matrix(
            y_true,
            df["ml_anomaly"]
        )
    )

ISOLATION FOREST EVALUATION

              precision    recall  f1-score   support

           0      0.894     0.967     0.929     12901
           1      0.652     0.348     0.453      2279

    accuracy                          0.874     15180
   macro avg      0.773     0.657     0.691     15180
weighted avg      0.857     0.874     0.858     15180

CONFUSION MATRIX:
[[12478   423]
 [ 1487   792]]


In [42]:
if y_true is not None:
    
    thresholds = np.arange(0.30, 0.81, 0.05)

    threshold_results = []

    for threshold in thresholds:

        prediction = (
            df["final_risk_score"] >= threshold
        ).astype(int)

        threshold_results.append({
            "threshold": round(threshold, 2),

            "precision": precision_score(
                y_true,
                prediction,
                zero_division=0
            ),

            "recall": recall_score(
                y_true,
                prediction,
                zero_division=0
            ),

            "f1_score": f1_score(
                y_true,
                prediction,
                zero_division=0
            ),

            "anomalies_detected": int(
                prediction.sum()
            )
        })

    threshold_results_df = pd.DataFrame(
        threshold_results
    )

    display(threshold_results_df)

,threshold,precision,recall,f1_score,anomalies_detected
0,0.30,0.612281,0.306275,0.408307,1140
1,0.35,0.682692,0.155770,0.253662,520
2,0.40,0.629771,0.072400,0.129870,262
3,0.45,0.722222,0.028521,0.054875,90
4,0.50,0.727273,0.014041,0.027551,44
5,0.55,0.619048,0.005704,0.011304,21
6,0.60,0.583333,0.003072,0.006111,12
7,0.65,0.750000,0.002633,0.005247,8
8,0.70,1.000000,0.001755,0.003504,4
9,0.75,1.000000,0.000439,0.000877,1


In [43]:
if y_true is not None:
    
    best_row = threshold_results_df.loc[
        threshold_results_df["f1_score"].idxmax()
    ]

    BEST_THRESHOLD = best_row["threshold"]

    print("BEST THRESHOLD:", BEST_THRESHOLD)
    print("PRECISION:", best_row["precision"])
    print("RECALL:", best_row["recall"])
    print("BEST F1 SCORE:", best_row["f1_score"])

else:
    BEST_THRESHOLD = 0.55

BEST THRESHOLD: 0.3
PRECISION: 0.612280701754386
RECALL: 0.30627468187801665
BEST F1 SCORE: 0.4083065223749634


In [44]:
df["final_anomaly"] = (
    df["final_risk_score"] >= BEST_THRESHOLD
).astype(int)

print("FINAL ANOMALY DISTRIBUTION:")
print(df["final_anomaly"].value_counts())

FINAL ANOMALY DISTRIBUTION:
final_anomaly
0    14040
1     1140
Name: count, dtype: int64


In [45]:
if y_true is not None:
    
    print("FINAL HYBRID MODEL EVALUATION\n")

    print(
        classification_report(
            y_true,
            df["final_anomaly"],
            digits=3
        )
    )

    print("FINAL CONFUSION MATRIX:")

    print(
        confusion_matrix(
            y_true,
            df["final_anomaly"]
        )
    )

FINAL HYBRID MODEL EVALUATION

              precision    recall  f1-score   support

           0      0.887     0.966     0.925     12901
           1      0.612     0.306     0.408      2279

    accuracy                          0.867     15180
   macro avg      0.750     0.636     0.667     15180
weighted avg      0.846     0.867     0.847     15180

FINAL CONFUSION MATRIX:
[[12459   442]
 [ 1581   698]]


In [46]:
def get_issue_description(row):
    
    issues = []

    for col in rule_columns:
        if row[col] == 1:
            issues.append(
                col.replace("rule_", "")
                   .replace("_", " ")
            )

    if row["ml_anomaly"] == 1:
        issues.append(
            "ML detected unusual authorization pattern"
        )

    if row["cluster_risk_score"] >= 0.75:
        issues.append(
            "Unusual behavioral pattern within cluster"
        )

    if len(issues) == 0:
        return "No issue detected"

    return "; ".join(issues)


df["detected_issues"] = df.apply(
    get_issue_description,
    axis=1
)

In [47]:
result_columns = [
    "authorization_id",
    "member_id",
    "provider_id",
    "authorization_date",
    "service_date",
    "authorization_status",
    "processing_time_hours",
    "missing_document_count",
    "resubmission_count",
    "rule_score",
    "rule_severity",
    "ml_anomaly",
    "ml_risk_score",
    "behavior_cluster",
    "cluster_risk_score",
    "final_risk_score",
    "final_severity",
    "final_anomaly",
    "sla",
    "detected_issues"
]

df[result_columns].sort_values(
    "final_risk_score",
    ascending=False
).head(30)

,authorization_id,member_id,provider_id,authorization_date,service_date,authorization_status,processing_time_hours,missing_document_count,resubmission_count,rule_score,rule_severity,ml_anomaly,ml_risk_score,behavior_cluster,cluster_risk_score,final_risk_score,final_severity,final_anomaly,sla,detected_issues
2993,AUTH01519,NaN,prv0353,2026-06-11,2026-06-13,Denied,NaN,1,0,3,Medium,1,0.741163,2,0.999989,0.796021,Critical,1,Immediate Review within 4 Hours,missing member id; invalid member id; invalid ...
8606,AUTH90072,MEM03701,PRV0050,2026-10-10,2025-12-12,Approved,98.98,1,4,3,Medium,1,0.917728,0,0.268853,0.729248,High,1,Review within 24 Hours,future authorization date; invalid date sequen...
13104,AUTH12066,MEM03701,PRV0050,2026-10-10,2025-12-12,Approved,98.98,1,3,3,Medium,1,0.887214,0,0.268476,0.715442,High,1,Review within 24 Hours,future authorization date; invalid date sequen...
5651,AUTH04512,MEM04767,0186,2026-08-18,2026-08-26,Approved,NaN,1,0,2,Medium,1,0.746158,2,1.000000,0.710771,High,1,Review within 24 Hours,invalid provider id; future service date; ML d...
3454,AUTH08887,MEM04647,PRV0380,2026-10-02,2025-12-22,Pending,493.31,0,0,2,Medium,1,1.000000,0,0.278138,0.680628,High,1,Review within 24 Hours,future authorization date; invalid date sequen...
4335,AUTH10483,MEM01990,PRV0487,2026-09-11,2025-11-26,Approved,-17.91,0,0,3,Medium,1,0.779298,1,0.259550,0.665094,High,1,Review within 24 Hours,future authorization date; invalid date sequen...
6470,AUTH05201,NaN,PRV0050,2026-08-17,2026-08-20,Pending,15.07,0,2,4,Critical,1,0.649074,0,0.061358,0.654355,High,1,Review within 24 Hours,missing member id; invalid member id; future s...
1834,AUTH01761,MEM03557,PRV0185,2026-10-10,2026-02-22,Approved,379.07,0,0,2,Medium,1,0.967026,0,0.216524,0.653467,High,1,Review within 24 Hours,future authorization date; invalid date sequen...
2130,AUTH11812,MEM04844,PRV0483,2026-09-18,2025-10-26,Denied,43.31,0,9,3,Medium,1,0.684203,1,0.290862,0.628564,High,1,Review within 24 Hours,future authorization date; invalid date sequen...
9549,AUTH14452,MEM01971,PRVX0295,2026-07-18,2026-07-19,Pending,NaN,1,0,1,Low,1,0.742174,2,0.999990,0.621476,High,1,Review within 24 Hours,invalid provider id; ML detected unusual autho...


In [48]:
OUTPUT_PATH = "authorization_anomaly_results.csv"

df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Results saved successfully!")
print("Output file:", OUTPUT_PATH)
print("Final Dataset Shape:", df.shape)

Results saved successfully!
Output file: authorization_anomaly_results.csv
Final Dataset Shape: (15180, 66)


In [49]:
feature_reference = {}

for feature in ml_features:

    values = df.loc[
        df["final_anomaly"] == 0,
        feature
    ].dropna()

    if len(values) == 0:
        values = df[feature].dropna()

    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)

    feature_reference[feature] = {
        "median": float(values.median()),
        "lower": float(q1 - 1.5 * (q3 - q1)),
        "upper": float(q3 + 1.5 * (q3 - q1))
    }

print(feature_reference)

{'processing_time_hours': {'median': 23.94, 'lower': -15.036249999999999, 'upper': 65.37375}, 'missing_document_count': {'median': 0.0, 'lower': -1.5, 'upper': 2.5}, 'resubmission_count': {'median': 0.0, 'lower': -1.5, 'upper': 2.5}, 'authorization_to_service_days': {'median': 3.0, 'lower': -10.5, 'upper': 17.5}, 'provider_avg_processing_time': {'median': 29.78375, 'lower': 7.506867201426022, 'upper': 55.9669385026738}, 'provider_avg_resubmission': {'median': 0.6470588235294118, 'lower': -0.18716577540106955, 'upper': 1.5240641711229945}, 'provider_avg_missing_docs': {'median': 0.7, 'lower': 0.04743833017077792, 'upper': 1.3833017077798861}, 'processing_time_provider_deviation': {'median': 12.546396103896104, 'lower': -17.04344642857143, 'upper': 44.31907738095238}}


In [50]:
def get_ml_evidence(row, top_n=3):
    
    evidence = []

    for feature in ml_features:

        value = row[feature]

        if pd.isna(value):
            continue

        reference = feature_reference[feature]

        lower = reference["lower"]
        upper = reference["upper"]
        median = reference["median"]

        if value > upper:

            deviation_score = (
                (value - upper) /
                (abs(upper - median) + 1e-6)
            )

            evidence.append({
                "feature": feature,
                "observed_value": round(float(value), 2),
                "expected_lower": round(float(lower), 2),
                "expected_upper": round(float(upper), 2),
                "direction": "above_normal",
                "deviation_score": round(float(deviation_score), 3)
            })

        elif value < lower:

            deviation_score = (
                (lower - value) /
                (abs(median - lower) + 1e-6)
            )

            evidence.append({
                "feature": feature,
                "observed_value": round(float(value), 2),
                "expected_lower": round(float(lower), 2),
                "expected_upper": round(float(upper), 2),
                "direction": "below_normal",
                "deviation_score": round(float(deviation_score), 3)
            })

    evidence = sorted(
        evidence,
        key=lambda x: x["deviation_score"],
        reverse=True
    )

    return evidence[:top_n]

In [51]:
def get_rule_evidence(row):
    
    findings = []

    for rule in rule_columns:

        if row[rule] == 1:

            rule_name = rule.replace("rule_", "")

            findings.append({
                "rule_name": rule_name,
                "status": "violated"
            })

    return findings

In [52]:
df["rule_based_findings"] = df.apply(
    get_rule_evidence,
    axis=1
)

df["ml_based_findings"] = None

ml_mask = df["ml_anomaly"] == 1

df.loc[ml_mask, "ml_based_findings"] = df.loc[
    ml_mask
].apply(
    lambda row: {
        "model": "Isolation Forest",
        "is_anomaly": True,
        "anomaly_score": round(
            float(row["ml_risk_score"]),
            4
        ),
        "contributing_features": get_ml_evidence(row)
    },
    axis=1
)

In [53]:
import json

In [54]:
def clean_json_value(value):
    
    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d")

    if pd.isna(value):
        return None

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return round(float(value), 4)

    if isinstance(value, np.bool_):
        return bool(value)

    return value

In [58]:
def build_rag_record(row):
    
    return {
        "dataset_type": "authorization",

        "record_id": clean_json_value(
            row["authorization_id"]
        ),

        "detection_summary": {
    "final_anomaly": bool(row["final_anomaly"]),

    "final_severity": clean_json_value(
        row["final_severity"]
    ),

    # Overall final risk score
    "final_risk_score": round(
        float(row["final_risk_score"]), 4
    ),

    # Individual risk components
    "rule_risk_score": round(
        float(row["rule_risk_score"]), 4
    ),

    "ml_risk_score": round(
        float(row["ml_risk_score"]), 4
    ),

    "cluster_risk_score": round(
        float(row["cluster_risk_score"]), 4
    ),

    "rule_anomaly": bool(row["rule_anomaly"]),

    "ml_anomaly": bool(row["ml_anomaly"])
},

        "rule_based_evidence": row[
            "rule_based_findings"
        ],

        "ml_based_evidence": row[
            "ml_based_findings"
        ],

        "record_context": {
            "authorization_date": clean_json_value(
                row["authorization_date"]
            ),

            "service_date": clean_json_value(
                row["service_date"]
            ),

            "service_type": clean_json_value(
                row["service_type"]
            ),

            "procedure_code": clean_json_value(
                row["procedure_code"]
            ),

            "authorization_status": clean_json_value(
                row["authorization_status"]
            ),

            "authorization_type": clean_json_value(
                row["authorization_type"]
            ),

            "urgency": clean_json_value(
                row["urgency"]
            ),

            "submission_channel": clean_json_value(
                row["submission_channel"]
            ),

            "processing_time_hours": clean_json_value(
                row["processing_time_hours"]
            ),

            "missing_document_count": clean_json_value(
                row["missing_document_count"]
            ),

            "resubmission_count": clean_json_value(
                row["resubmission_count"]
            )
        },

        "sla": clean_json_value(row["sla"])
    }

In [59]:
rag_records = []

anomaly_df = df[
    df["final_anomaly"] == 1
].copy()

for _, row in anomaly_df.iterrows():

    rag_records.append(
        build_rag_record(row)
    )

print("Records prepared for RAG:", len(rag_records))

Records prepared for RAG: 1140


In [60]:
OUTPUT_JSON = "authorization_anomalies_for_rag.json"

with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        rag_records,
        f,
        indent=4,
        ensure_ascii=False,
        default=str
    )

print("RAG JSON saved successfully!")
print("File:", OUTPUT_JSON)

RAG JSON saved successfully!
File: authorization_anomalies_for_rag.json
